# Figure 3 — Hippo signaling is required for efficient endothelial differentiation

This notebook reproduces computational panels **b, c, d, and f**.

- **Panel a** is a pathway schematic and is intentionally omitted.
- **Panel b** reuses the recovered Figure 2e feature-UMAP style and Figure 2f carrier-seeded neighborhood workflow for P2, P34, CDH5, ACTC1, NF2, LATS2, AMOTL2, and TAOK1.
- **Panel c** recalculates guide-level Day 3 EC percentages from the integrated CC-Perturb-seq object. The control distribution comprises all 120 negative-control guides (60 non-targeting plus 60 safe-targeting). Control-guide carriers that also carry any guide targeting NF2, LATS2, AMOTL2, or TAOK1 are excluded. Two-sided Welch tests are BH-adjusted across the four displayed genes.
- **Panel d** ports the focused program-annotation workflow for D3 P2/P34: top three program-specific regulators, exact manuscript marker genes, and the top five motifs per program from the minimum-10-carrier motif table. Motif logos are resolved from the cluster map and compiled MoDISco H5.
- **Panel e** is an experimental flow-cytometry panel and is intentionally omitted.
- **Panel f** ports the attached Hippo time-course dotplot workflow, but reads the query and `top_specific_200` cNMF reprojection paths established by `reproject_cnmf_compare_gene_sets.ipynb`.

Panels b and c use **plotnine**. Panel d retains the recovered vector Matplotlib annotation implementation because that source notebook used Matplotlib to align heatmaps and rotated motif logos. Panel f retains Scanpy's dotplot implementation from the attached source notebook.

In [ ]:
# Repository-path bootstrap for local support modules.
from pathlib import Path
import sys


def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing "
        "src/cc_perturbseq. Run this notebook from within the "
        "cc-perturbseq-manuscript-analysis checkout."
    )


REPO_ROOT = _find_repo_root()
for _support_path in (REPO_ROOT, REPO_ROOT / "src"):
    _support_path_text = str(_support_path)
    if _support_path_text not in sys.path:
        sys.path.insert(0, _support_path_text)

print(
    "Repository support paths added:",
    REPO_ROOT,
    "and",
    REPO_ROOT / "src",
)


In [ ]:
from IPython.display import display
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.cc_perturbseq.figure_notebook import (
    get_path,
    initialize_figure,
    panel_directory,
    sha256_file,
    write_receipt,
)
from src.cc_perturbseq.figure3_panels import (
    FIGURE3_HIPPO_TARGETS,
    compute_figure3b_carrier_seeded,
    compute_figure3c_guide_ec_120_controls,
    make_figure3d_annotation,
    plot_figure3b_carrier_seeded,
    plot_figure3b_feature_umaps,
    plot_figure3c_120_controls,
    prepare_figure3f_dotplots,
)
from src.cc_perturbseq.program_panels import load_integrated_anndata

EXPECTED_WORKBOOK_SHA256 = (
    "d2fe4834b1efd3c00e01062b56b907a6178bfbc48a40beb5ea7b347538bfc824"
)


def export_plotnine(plot, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix in ("pdf", "svg", "png"):
        path = out / f"{stem}.{suffix}"
        plot.save(filename=path, dpi=300, verbose=False)
        result[suffix] = str(path)
    return result


def export_matplotlib(figure, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix in ("pdf", "svg", "png"):
        path = out / f"{stem}.{suffix}"
        kwargs = {"dpi": 300} if suffix == "png" else {}
        figure.savefig(
            path,
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
            **kwargs,
        )
        result[suffix] = str(path)
    return result


def export_scanpy_dotplot(dotplot, panel, stem):
    out = panel_directory(ctx, panel)
    result = {}
    for suffix in ("pdf", "svg", "png"):
        path = out / f"{stem}.{suffix}"
        dotplot.savefig(path)
        result[suffix] = str(path)
    return result


FIGURE = {
    "figure_id": "Figure_03",
    "title": "Figure 3 — Hippo signaling is required for efficient endothelial differentiation",
    "scope": "Panels b–d and f; panels a and e omitted",
    "manuscript": "Working CC-Perturb-seq EC Differentiation - Text V3",
    "panels": [
        {
            "panel": "b",
            "description": "P2/P34, CDH5/ACTC1, and Hippo carrier-neighborhood UMAPs",
            "type": "code",
            "required_inputs": [
                "ccperturbseq.integrated_anndata",
                "ccperturbseq.cnmf_usage",
                "ccperturbseq.target_metadata",
            ],
        },
        {
            "panel": "c",
            "description": "Guide-level Day 3 EC fractions for four Hippo-pathway targets",
            "type": "code",
            "required_inputs": [
                "ccperturbseq.integrated_anndata",
                "ccperturbseq.target_metadata",
            ],
        },
        {
            "panel": "d",
            "description": "Integrated P2/P34 regulator, loading, and motif annotation",
            "type": "mixed",
            "required_inputs": [
                "ccperturbseq.cnmf_spectra",
                "multiome.enhancer_enrichment_results",
                "multiome.cluster_motif_map",
                "multiome.modisco_compiled_h5",
            ],
        },
        {
            "panel": "f",
            "description": "Hippo-modulator marker-gene and projected-program dotplots",
            "type": "code",
            "required_inputs": [
                "hippo_timecourse.query_h5ad",
                "hippo_timecourse.reprojected_cnmf_h5ad",
            ],
        },
    ],
}
ctx = initialize_figure(FIGURE)

workbook = (
    ctx["root"]
    / "data"
    / "figure_02"
    / "cNMF_50_0_2_032526_6co.xlsx"
)
workbook_hash = sha256_file(workbook) if workbook.exists() else None
if workbook_hash != EXPECTED_WORKBOOK_SHA256:
    raise ValueError(
        "Missing or incorrect repository-local cNMF workbook at "
        f"{workbook}. Observed SHA-256={workbook_hash}; "
        f"expected {EXPECTED_WORKBOOK_SHA256}."
    )

adata_path = get_path(ctx["paths"], "ccperturbseq.integrated_anndata")
usage_path = get_path(ctx["paths"], "ccperturbseq.cnmf_usage")
guide_metadata_path = get_path(ctx["paths"], "ccperturbseq.target_metadata")
adata, usage = load_integrated_anndata(adata_path, usage_path)

ctx["input_audit"]

## Panel b — P2/P34, marker genes, and carrier-seeded neighborhoods

The feature maps reuse the Figure 2e plotnine implementation. Carrier-seeded neighborhoods reuse the recovered Figure 2f workflow: all target carriers are seeds; all 120 pooled negative-control guides define target-specific controls; neighbors are restricted to the same differentiation day; and enrichment is computed in 300-neighbor Euclidean neighborhoods in z-scored 50-program space.

In [ ]:
plot_b_features, source_b_features = plot_figure3b_feature_umaps(
    adata,
    usage,
)
scores_b, background_b = compute_figure3b_carrier_seeded(
    adata,
    usage,
    guide_metadata_path,
    targets=FIGURE3_HIPPO_TARGETS,
    k=300,
    pseudocount=0.5,
)
plot_b_carriers, scores_b_plot, background_b_plot = (
    plot_figure3b_carrier_seeded(
        scores_b,
        background_b,
        targets=FIGURE3_HIPPO_TARGETS,
    )
)

out = panel_directory(ctx, "b")
feature_outputs = export_plotnine(
    plot_b_features,
    "b",
    "Figure_03b_P2_P34_CDH5_ACTC1_umaps",
)
carrier_outputs = export_plotnine(
    plot_b_carriers,
    "b",
    "Figure_03b_Hippo_carrier_seeded_neighborhoods",
)

source_paths = {}
for name, table in source_b_features.items():
    path = out / f"Figure_03b_{name}_source.csv.gz"
    table.to_csv(path, compression="gzip")
    source_paths[name] = str(path)
scores_path = out / "Figure_03b_carrier_seeded_scores.csv.gz"
background_path = out / "Figure_03b_carrier_background_cells.csv.gz"
scores_b.to_csv(scores_path, index=False, compression="gzip")
background_b.to_csv(background_path, index=False, compression="gzip")

write_receipt(
    ctx,
    "b",
    status="SOURCE_WORKFLOW_PORTED_REQUIRES_SHERLOCK_EXECUTION",
    inputs={
        "anndata": str(adata_path),
        "usage": str(usage_path),
        "guide_metadata": str(guide_metadata_path),
    },
    outputs={
        **{f"feature_{k}": v for k, v in feature_outputs.items()},
        **{f"carrier_{k}": v for k, v in carrier_outputs.items()},
        **source_paths,
        "carrier_scores": str(scores_path),
        "carrier_background": str(background_path),
    },
    parameters={
        "features": ["P2", "CDH5", "P34", "ACTC1"],
        "carrier_targets": list(FIGURE3_HIPPO_TARGETS),
        "control_guides": "60 non-targeting + 60 safe-targeting",
        "neighbors": 300,
        "same_day_neighbors": True,
        "distance": "Euclidean on z-scored 50-program usage",
        "pseudocount": 0.5,
        "plotting_backend": "plotnine",
    },
)
display(plot_b_features)
display(plot_b_carriers)

## Panel c — guide-level Day 3 EC fractions

For each of the 120 individual control guides, D3 cells carrying any guide targeting NF2, LATS2, AMOTL2, or TAOK1 are excluded before its EC percentage is calculated. Targeting-guide percentages retain all D3 carriers of the corresponding guide. Each target-guide distribution is compared with the common 120-guide control distribution using a two-sided Welch's t-test; the four P values are BH-adjusted.

In [ ]:
guide_c, stats_c, meta_c = compute_figure3c_guide_ec_120_controls(
    adata,
    guide_metadata_path,
    targets=FIGURE3_HIPPO_TARGETS,
)
plot_c, jitter_c = plot_figure3c_120_controls(
    guide_c,
    stats_c,
    targets=FIGURE3_HIPPO_TARGETS,
)

out = panel_directory(ctx, "c")
outputs_c = export_plotnine(
    plot_c,
    "c",
    "Figure_03c_guide_level_D3_EC_fraction",
)
guide_path = out / "Figure_03c_guide_level_D3_EC_percentages.csv"
stats_path = out / "Figure_03c_Welch_statistics.csv"
jitter_path = out / "Figure_03c_plot_jitter_coordinates.csv"
guide_c.to_csv(guide_path, index=False)
stats_c.to_csv(stats_path, index=False)
jitter_c.to_csv(jitter_path, index=False)

write_receipt(
    ctx,
    "c",
    status="SOURCE_VERIFIED_METHOD_IMPLEMENTATION_REQUIRES_SHERLOCK_EXECUTION",
    inputs={
        "anndata": str(adata_path),
        "guide_metadata": str(guide_metadata_path),
    },
    outputs={
        **outputs_c,
        "guide_percentages": str(guide_path),
        "welch_statistics": str(stats_path),
        "jitter_coordinates": str(jitter_path),
    },
    parameters={
        "targets": list(FIGURE3_HIPPO_TARGETS),
        "control_guides": "120: 60 non-targeting + 60 safe-targeting",
        "control_exclusion_targets": list(FIGURE3_HIPPO_TARGETS),
        "targeting_guide_exclusion": "none beyond D3 restriction",
        "test": "two-sided Welch t-test",
        "BH_tests": 4,
        "point_unit": "guide",
        "plotting_backend": "plotnine",
        **meta_c,
    },
)
display(plot_c)
stats_c

## Panel d — integrated annotation of P2 and P34

This cell ports the recovered focused program-annotation notebook. It uses the repository-local cNMF workbook, the final 030526 consensus spectra, the new minimum-10-carrier motif-enrichment table, and the established motif cluster map and compiled MoDISco H5.

For each program it displays:

1. the first three genes from `Top 5 specific regulators (FDR<0.1) D3`;
2. the exact manuscript marker-gene list;
3. the top five motifs ranked by q value, regardless of significance.

The top-five motif union is exported so that the final three columns can be selected in Illustrator, as in the original figure assembly.

In [ ]:
spectra_path = get_path(ctx["paths"], "ccperturbseq.cnmf_spectra")
motif_path = get_path(ctx["paths"], "multiome.enhancer_enrichment_results")
cluster_map_path = get_path(ctx["paths"], "multiome.cluster_motif_map")
modisco_h5_path = get_path(ctx["paths"], "multiome.modisco_compiled_h5")

figure_d, source_d, meta_d = make_figure3d_annotation(
    workbook_path=workbook,
    spectra_path=spectra_path,
    motif_path=motif_path,
    cluster_map_path=cluster_map_path,
    modisco_h5_path=modisco_h5_path,
    top_motifs_per_program=5,
)

out = panel_directory(ctx, "d")
outputs_d = export_matplotlib(
    figure_d,
    "d",
    "Figure_03d_P2_P34_program_annotation_top5_motifs",
)
source_paths_d = {}
for name, table in source_d.items():
    path = out / f"Figure_03d_{name}.csv"
    table.to_csv(path)
    source_paths_d[name] = str(path)

write_receipt(
    ctx,
    "d",
    status="RECOVERED_PROGRAM_ANNOTATION_WORKFLOW_REQUIRES_SHERLOCK_EXECUTION",
    inputs={
        "workbook": str(workbook),
        "spectra": str(spectra_path),
        "motif_enrichment": str(motif_path),
        "motif_cluster_map": str(cluster_map_path),
        "modisco_compiled_h5": str(modisco_h5_path),
    },
    outputs={**outputs_d, **source_paths_d},
    parameters={
        "programs": ["P2", "P34"],
        "markers": {
            "P2": ["CDH5", "NRP1", "EFNB2"],
            "P34": ["TAGLN", "KRT8", "ACTC1", "ANKRD1"],
        },
        "top_regulators_per_program": 3,
        "top_motifs_per_program": 5,
        "motif_significance_required_for_selection": False,
        "motif_color": "-log10(q-value)",
        "motif_logo_rotation_degrees": 90,
        "plotting_backend": "recovered vector Matplotlib/logomaker workflow",
        **meta_d,
    },
)
plt.show()
source_d["motif_selection"]

## Panel f — Hippo-modulator marker and projected-program dotplots

The attached `hippo_tc_analysis_clean.ipynb` is retained as the plotting/statistical source. Its placeholder H5AD paths are replaced by the query and `top_specific_200` reprojection paths established in `reproject_cnmf_compare_gene_sets.ipynb`.

The five Day 3 conditions, condition ordering, marker genes, per-gene `standard_scale='var'`, and cNMF CLR dotplot settings are unchanged from the attached notebook.

In [ ]:
query_h5ad_path = get_path(ctx["paths"], "hippo_timecourse.query_h5ad")
reprojected_h5ad_path = get_path(
    ctx["paths"],
    "hippo_timecourse.reprojected_cnmf_h5ad",
)

(
    gene_dotplot_f,
    program_dotplot_f,
    gene_source_f,
    program_source_f,
    meta_f,
) = prepare_figure3f_dotplots(
    query_h5ad_path,
    reprojected_h5ad_path,
)

out = panel_directory(ctx, "f")
gene_outputs_f = export_scanpy_dotplot(
    gene_dotplot_f,
    "f",
    "Figure_03f_D3_marker_gene_dotplot",
)
program_outputs_f = export_scanpy_dotplot(
    program_dotplot_f,
    "f",
    "Figure_03f_D3_projected_cNMF_dotplot",
)
gene_source_path = out / "Figure_03f_marker_gene_source.csv"
program_source_path = out / "Figure_03f_program_CLR_source.csv"
gene_source_f.to_csv(gene_source_path, index=False)
program_source_f.to_csv(program_source_path, index=False)

write_receipt(
    ctx,
    "f",
    status="ATTACHED_NOTEBOOK_WORKFLOW_PORTED_REQUIRES_SHERLOCK_EXECUTION",
    inputs={
        "query_h5ad": str(query_h5ad_path),
        "reprojected_cnmf_h5ad": str(reprojected_h5ad_path),
    },
    outputs={
        **{f"genes_{k}": v for k, v in gene_outputs_f.items()},
        **{f"programs_{k}": v for k, v in program_outputs_f.items()},
        "gene_source": str(gene_source_path),
        "program_source": str(program_source_path),
    },
    parameters={
        "conditions": [
            "D3 Control",
            "D2-3 HIPPOa",
            "D1-3 HIPPOa",
            "D2-3 HIPPOi",
            "D1-3 HIPPOi",
        ],
        "genes": ["CDH5", "NRP1", "EFNB2", "KRT8", "TAGLN", "ANKRD1"],
        "programs": ["P2", "P34"],
        "gene_scaling": "Scanpy standard_scale='var'",
        "program_layer": "X_cnmf_clr_top_specific_200",
        "program_projection_gene_set": "top_specific_200",
        "plotting_backend": "Scanpy dotplot",
        **meta_f,
    },
)
gene_dotplot_f.show()
program_dotplot_f.show()
gene_source_f, program_source_f